> VIGOR 2.12.1 example. Install with `pip install "VIGOR[io,progress,plot] @ git+https://github.com/sanchestm/VIGOR"` (see the [README](../../../README.md)). Outputs are cleared: this notebook was checked against the 2.12.1 API but not re-executed for this release.

# Covariance mediation and PC/ANM/SEM

VIGOR 2.12.1. Expression mediation needs a separately aligned expression matrix; this executes a trait covariance screen without fabricating expression data.


In [ ]:
from pathlib import Path
import sys, os, json, warnings
import numpy as np
import pandas as pd
import xarray as xr

# Find ROOT/vigor_run/project_config.py regardless of notebook cwd.
def _find_run(start=None):
    p=Path(os.environ.get('VIGOR_PROJECT_ROOT', os.environ.get('DOMINO_PROJECT_ROOT', Path.cwd()))) if start is None else Path(start)
    for q in [p,*p.parents]:
        cand=q/'vigor_run'/'project_config.py'
        if cand.exists(): return cand.parent
        if q.name=='vigor_run' and (q/'project_config.py').exists(): return q
    raise FileNotFoundError('Set VIGOR_PROJECT_ROOT or place vigor_run under the project root.')
RUN=_find_run(); sys.path.insert(0,str(RUN))
from project_config import *
import vigor
print('Project root:',ROOT)
print('VIGOR:',vigor.__version__)
assert tuple(map(int, vigor.__version__.split('.')[:3])) >= (2, 12, 1), vigor.__version__


In [ ]:
train=pd.read_csv(TRAIN_CSV,index_col=0);train.index=train.index.astype(str)
store=vigor.open_store(ANALYSIS_STORE,mode="a",lazy=False)
cov=store.open_dataset("covariance/GE_full",lazy=False)
G=np.asarray(cov.G,float);E=np.asarray(cov.E,float)
if G.ndim==3:G=G.sum(axis=0)


In [ ]:
from vigor.causal import genetic_covariance_mediation,pc_from_covariance,anm_pairwise_direction,fit_multisource_covariance_sem
names=list(train.columns);p=min(4,len(names))
mediation=genetic_covariance_mediation(G,[0],outcomes=list(range(1,p)),E=E,names=names)
print("conditional mediation:",mediation.keys())
# Data-driven graph discovery is restricted to a small module.
graph=pc_from_covariance(G[:p,:p],names=names[:p],n_eff=int(train.notna().any(axis=1).sum()),max_cond_set=1)
print("PC:",type(graph).__name__)
print("ANM:",anm_pairwise_direction(train.iloc[:,0],train.iloc[:,1],min_samples=40))
# The order below is a declared modeling assumption, not an inferred causal conclusion.
sem=fit_multisource_covariance_sem({"G":G[:p,:p],"E":E[:p,:p]},order=names[:p],names=names[:p],maxiter=100)
print("SEM B:",np.asarray(sem["B"].to_pandas()).shape)
